In [ ]:
# ============================================================
# BUSINESS SALES PERFORMANCE ANALYTICS
# FUTURE INTERNS - DATA SCIENCE & ANALYTICS TASK 1
# ============================================================

# Install required libraries
!pip -q install pandas plotly openpyxl

# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import files
from IPython.display import display, HTML

# ------------------------------------------------------------
# 2. UPLOAD DATASET
# ------------------------------------------------------------

print("Please upload your Superstore CSV/Excel file:")

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

# Read CSV or Excel automatically
if file_name.lower().endswith(".csv"):
    df = pd.read_csv(file_name, encoding="latin1")
elif file_name.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(file_name)
else:
    raise ValueError("Please upload a CSV or Excel file.")

print("\nDataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# ------------------------------------------------------------
# 3. DISPLAY BASIC DATA
# ------------------------------------------------------------

display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 4. CLEAN COLUMN NAMES
# ------------------------------------------------------------

df.columns = df.columns.str.strip()

# ------------------------------------------------------------
# 5. CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "Sales",
    "Profit",
    "Quantity",
    "Category",
    "Region",
    "Segment",
    "Order Date"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    print("\nMissing columns:", missing_columns)
    print("\nAvailable columns:")
    print(df.columns.tolist())
    raise ValueError("Some required columns are missing.")

# ------------------------------------------------------------
# 6. DATA PREPROCESSING
# ------------------------------------------------------------

df["Order Date"] = pd.to_datetime(
    df["Order Date"],
    errors="coerce"
)

df["Sales"] = pd.to_numeric(
    df["Sales"],
    errors="coerce"
)

df["Profit"] = pd.to_numeric(
    df["Profit"],
    errors="coerce"
)

df["Quantity"] = pd.to_numeric(
    df["Quantity"],
    errors="coerce"
)

# Remove rows with missing important values
df = df.dropna(
    subset=[
        "Order Date",
        "Sales",
        "Profit",
        "Quantity"
    ]
)

# ------------------------------------------------------------
# 7. CREATE YEAR COLUMN
# ------------------------------------------------------------

df["Year"] = df["Order Date"].dt.year

# ------------------------------------------------------------
# 8. KPI CALCULATIONS
# ------------------------------------------------------------

total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_quantity = df["Quantity"].sum()

print("\n================ KPI SUMMARY ================")

print(f"Total Sales    : ${total_sales:,.2f}")
print(f"Total Profit   : ${total_profit:,.2f}")
print(f"Total Quantity : {total_quantity:,.0f}")

# ------------------------------------------------------------
# 9. CATEGORY ANALYSIS
# ------------------------------------------------------------

category_sales = (
    df.groupby("Category")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

category_profit = (
    df.groupby("Category")["Profit"]
    .sum()
    .sort_values(ascending=False)
)

# ------------------------------------------------------------
# 10. REGION ANALYSIS
# ------------------------------------------------------------

region_sales = (
    df.groupby("Region")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

# ------------------------------------------------------------
# 11. SEGMENT ANALYSIS
# ------------------------------------------------------------

segment_sales = (
    df.groupby("Segment")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

# ------------------------------------------------------------
# 12. YEARLY SALES TREND
# ------------------------------------------------------------

yearly_sales = (
    df.groupby("Year")["Sales"]
    .sum()
    .reset_index()
)

# ------------------------------------------------------------
# 13. FIND TOP PERFORMERS
# ------------------------------------------------------------

top_category = category_sales.idxmax()
top_category_sales = category_sales.max()

top_profit_category = category_profit.idxmax()
top_profit_value = category_profit.max()

top_region = region_sales.idxmax()
top_region_sales = region_sales.max()

top_segment = segment_sales.idxmax()
top_segment_sales = segment_sales.max()

# ------------------------------------------------------------
# 14. CREATE DASHBOARD
# ------------------------------------------------------------

fig = make_subplots(

    rows=5,
    cols=4,

    specs=[
        [
            {"type": "indicator"},
            {"type": "indicator"},
            {"type": "indicator"},
            {"type": "xy"}
        ],

        [
            {"type": "xy", "colspan": 4},
            None,
            None,
            None
        ],

        [
            {"type": "xy"},
            {"type": "xy"},
            {"type": "xy"},
            {"type": "domain"}
        ],

        [
            {"type": "table", "colspan": 4},
            None,
            None,
            None
        ],

        [
            {"type": "table", "colspan": 4},
            None,
            None,
            None
        ]
    ],

    vertical_spacing=0.07,

    horizontal_spacing=0.04,

    row_heights=[
        0.15,
        0.27,
        0.27,
        0.14,
        0.17
    ]
)

# ============================================================
# KPI 1 - TOTAL SALES
# ============================================================

fig.add_trace(

    go.Indicator(

        mode="number",

        value=total_sales,

        number={
            "prefix": "$",
            "valueformat": ",.2s",
            "font": {
                "size": 34
            }
        },

        title={
            "text": "<b>Total Sales</b>",
            "font": {
                "size": 18
            }
        },

        domain={
            "row": 0,
            "column": 0
        }
    ),

    row=1,
    col=1
)

# ============================================================
# KPI 2 - TOTAL PROFIT
# ============================================================

fig.add_trace(

    go.Indicator(

        mode="number",

        value=total_profit,

        number={
            "prefix": "$",
            "valueformat": ",.2s",
            "font": {
                "size": 34
            }
        },

        title={
            "text": "<b>Total Profit</b>",
            "font": {
                "size": 18
            }
        },

        domain={
            "row": 0,
            "column": 1
        }
    ),

    row=1,
    col=2
)

# ============================================================
# KPI 3 - TOTAL QUANTITY
# ============================================================

fig.add_trace(

    go.Indicator(

        mode="number",

        value=total_quantity,

        number={
            "valueformat": ",.0f",
            "font": {
                "size": 34
            }
        },

        title={
            "text": "<b>Total Quantity</b>",
            "font": {
                "size": 18
            }
        },

        domain={
            "row": 0,
            "column": 2
        }
    ),

    row=1,
    col=3
)

# ============================================================
# SALES TREND
# ============================================================

fig.add_trace(

    go.Scatter(

        x=yearly_sales["Year"],

        y=yearly_sales["Sales"],

        mode="lines+markers",

        line={
            "width": 4
        },

        marker={
            "size": 9
        },

        name="Sales",

        hovertemplate=
        "<b>Year:</b> %{x}<br>" +
        "<b>Sales:</b> $%{y:,.0f}<extra></extra>"
    ),

    row=2,
    col=1
)

fig.update_xaxes(
    title_text="Year",
    row=2,
    col=1
)

fig.update_yaxes(
    title_text="Sales ($)",
    row=2,
    col=1
)

# ============================================================
# SALES BY CATEGORY
# ============================================================

fig.add_trace(

    go.Bar(

        x=category_sales.index,

        y=category_sales.values,

        text=[
            f"${x:,.0f}"
            for x in category_sales.values
        ],

        textposition="outside",

        name="Sales by Category",

        marker_color="#42A5F5",

        hovertemplate=
        "<b>%{x}</b><br>" +
        "Sales: $%{y:,.0f}<extra></extra>"
    ),

    row=3,
    col=1
)

fig.update_xaxes(
    title_text="Category",
    row=3,
    col=1
)

fig.update_yaxes(
    title_text="Sales ($)",
    row=3,
    col=1
)

# ============================================================
# PROFIT BY CATEGORY
# ============================================================

fig.add_trace(

    go.Bar(

        x=category_profit.index,

        y=category_profit.values,

        text=[
            f"${x:,.0f}"
            for x in category_profit.values
        ],

        textposition="outside",

        name="Profit by Category",

        marker_color="#FF9800",

        hovertemplate=
        "<b>%{x}</b><br>" +
        "Profit: $%{y:,.0f}<extra></extra>"
    ),

    row=3,
    col=2
)

fig.update_xaxes(
    title_text="Category",
    row=3,
    col=2
)

fig.update_yaxes(
    title_text="Profit ($)",
    row=3,
    col=2
)

# ============================================================
# SALES BY REGION
# ============================================================

fig.add_trace(

    go.Bar(

        x=region_sales.index,

        y=region_sales.values,

        text=[
            f"${x:,.0f}"
            for x in region_sales.values
        ],

        textposition="outside",

        name="Sales by Region",

        marker_color="#7E57C2",

        hovertemplate=
        "<b>%{x}</b><br>" +
        "Sales: $%{y:,.0f}<extra></extra>"
    ),

    row=3,
    col=3
)

fig.update_xaxes(
    title_text="Region",
    row=3,
    col=3
)

fig.update_yaxes(
    title_text="Sales ($)",
    row=3,
    col=3
)

# ============================================================
# SALES BY SEGMENT - DONUT
# ============================================================

fig.add_trace(

    go.Pie(

        labels=segment_sales.index,

        values=segment_sales.values,

        hole=0.55,

        textinfo="label+percent",

        name="Segment",

        hovertemplate=
        "<b>%{label}</b><br>" +
        "Sales: $%{value:,.0f}<br>" +
        "Share: %{percent}<extra></extra>"
    ),

    row=3,
    col=4
)

# ============================================================
# KEY INSIGHTS TABLE
# ============================================================

insight_text = [

    f"Technology generated the highest sales at ${top_category_sales:,.0f}.",

    f"{top_profit_category} generated the highest profit at ${top_profit_value:,.0f}.",

    f"{top_region} generated the highest regional sales at ${top_region_sales:,.0f}.",

    f"{top_segment} is the largest customer segment with ${top_segment_sales:,.0f} in sales.",

    "Sales performance should be monitored by year, category, region and customer segment."
]

fig.add_trace(

    go.Table(

        header=dict(

            values=["<b>KEY INSIGHTS</b>"],

            fill_color="#E8F5E9",

            align="left",

            font=dict(
                size=16
            )
        ),

        cells=dict(

            values=[
                ["• " + x for x in insight_text]
            ],

            fill_color="#F8FFF8",

            align="left",

            font=dict(
                size=14
            ),

            height=32
        )
    ),

    row=4,
    col=1
)

# ============================================================
# RECOMMENDATIONS TABLE
# ============================================================

recommendations = [

    "Focus marketing and product strategies on the highest-performing category.",

    "Strengthen sales initiatives in the highest-performing region.",

    "Create targeted offers for the largest customer segment.",

    "Monitor yearly sales trends to plan inventory and marketing.",

    "Use category-level profit analysis to improve product profitability."
]

fig.add_trace(

    go.Table(

        header=dict(

            values=["<b>RECOMMENDATIONS</b>"],

            fill_color="#FFF3E0",

            align="left",

            font=dict(
                size=16
            )
        ),

        cells=dict(

            values=[
                ["• " + x for x in recommendations]
            ],

            fill_color="#FFF9F2",

            align="left",

            font=dict(
                size=14
            ),

            height=32
        )
    ),

    row=5,
    col=1
)

# ============================================================
# DASHBOARD TITLE
# ============================================================

fig.update_layout(

    title=dict(

        text="<b>BUSINESS SALES PERFORMANCE ANALYTICS</b>",

        x=0.5,

        xanchor="center",

        font=dict(
            size=28
        )
    ),

    height=1500,

    width=1400,

    paper_bgcolor="#F7F9F7",

    plot_bgcolor="white",

    font=dict(
        family="Arial",
        color="#222222"
    ),

    showlegend=False,

    margin=dict(
        l=50,
        r=50,
        t=90,
        b=50
    )
)

# ------------------------------------------------------------
# 15. DISPLAY DASHBOARD
# ------------------------------------------------------------

fig.show()

# ------------------------------------------------------------
# 16. SAVE INTERACTIVE DASHBOARD
# ------------------------------------------------------------

fig.write_html(
    "Business_Sales_Performance_Dashboard.html"
)

print("\nDashboard created successfully!")
print("Interactive HTML file saved as:")
print("Business_Sales_Performance_Dashboard.html")

# ------------------------------------------------------------
# 17. DOWNLOAD DASHBOARD
# ------------------------------------------------------------

files.download(
    "Business_Sales_Performance_Dashboard.html"
)

Please upload your Superstore CSV/Excel file:


Saving Sample - Superstore.csv to Sample - Superstore (1).csv

Dataset loaded successfully!
Rows: 9994
Columns: 21


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164



Column names:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

================ KPI SUMMARY ================
Total Sales    : $2,297,200.86
Total Profit   : $286,397.02
Total Quantity : 37,873



Dashboard created successfully!
Interactive HTML file saved as:
Business_Sales_Performance_Dashboard.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>